# 00 · Colab 环境准备

本 Notebook 挂载 Google Drive、准备仓库与依赖、记录环境，并执行 import smoke。它优先保留 Colab 自带的 PyTorch；如果当前 Torch/CUDA 已兼容，不主动降级，也不强制替换 CUDA wheel。所有可编辑路径集中在下一格。

In [ ]:
from pathlib import Path

# 用户参数：按需修改这一格即可。
PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"
REPOSITORY_URL = "https://github.com/fish23611-beep/orthrus.git"
REPOSITORY_BRANCH = "feat/c8-experiment-matrix"
MOUNT_DRIVE = True
UPDATE_EXISTING_REPOSITORY = True
INSTALL_DEPENDENCIES = True
print({"project": str(PROJECT_ROOT), "artifacts": str(ARTIFACT_ROOT), "data": str(DATA_ROOT)})

In [ ]:
import os
import subprocess
import sys

if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        print("当前不是 Colab；跳过 Google Drive 挂载。")
    else:
        drive.mount("/content/drive")

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)
print("ORTHRUS_ARTIFACT_ROOT =", os.environ["ORTHRUS_ARTIFACT_ROOT"])
print("ORTHRUS_DATA_ROOT =", os.environ["ORTHRUS_DATA_ROOT"])

In [ ]:
import shutil

print("Python:", sys.version)
import torch
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
try:
    import torch_geometric
except ImportError:
    print("PyG: 尚未安装")
else:
    print("PyG:", torch_geometric.__version__)

if torch.cuda.is_available() and shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["df", "-h", str(PROJECT_ROOT.parent)], check=True)

In [ ]:
if (PROJECT_ROOT / ".git").is_dir():
    print("检测到现有仓库：", PROJECT_ROOT)
    if UPDATE_EXISTING_REPOSITORY:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        print("UPDATE_EXISTING_REPOSITORY=False，跳过更新。")
elif PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
    raise RuntimeError(f"项目路径已存在但不是 Git 仓库：{PROJECT_ROOT}")
else:
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--recurse-submodules", "--branch", REPOSITORY_BRANCH,
        REPOSITORY_URL, str(PROJECT_ROOT),
    ], check=True)
subprocess.run(["git", "-C", str(PROJECT_ROOT), "status", "-sb"], check=True)

## 安装策略

仓库 Dockerfile 是完整依赖的权威参考。下面只补齐非 Torch 依赖，并按当前 `torch.__version__` / `torch.version.cuda` 选择 PyG wheel 索引；不会卸载或降级 Colab 的 PyTorch。若 wheel 索引不支持当前组合，命令会明确失败，此时应选择兼容的 Colab runtime，而不是静默替换核心框架。

In [ ]:
if INSTALL_DEPENDENCIES:
    base_packages = [
        "scikit-learn", "networkx", "xxhash", "graphviz", "psutil", "matplotlib",
        "wandb", "chardet", "nltk", "igraph", "cairocffi", "wget", "gensim",
        "pytz", "pandas", "yacs", "psycopg2-binary", "tqdm", "pyyaml",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", *base_packages], check=True)

    torch_base = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    pyg_index = f"https://data.pyg.org/whl/torch-{torch_base}+{cuda_tag}.html"
    print("PyG wheel index:", pyg_index)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch_geometric"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "pyg_lib", "torch_scatter",
        "torch_sparse", "torch_cluster", "torch_spline_conv", "-f", pyg_index,
    ], check=True)
else:
    print("INSTALL_DEPENDENCIES=False，保留当前环境。")

In [ ]:
import importlib

src_root = str(PROJECT_ROOT / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)
for module_name in ("config", "orthrus", "torch", "torch_geometric", "pandas", "yaml"):
    module = importlib.import_module(module_name)
    print("import OK:", module_name, getattr(module, "__version__", ""))

In [ ]:
environment_dir = ARTIFACT_ROOT / "environment"
environment_dir.mkdir(parents=True, exist_ok=True)
freeze_path = environment_dir / "pip_freeze.txt"
with freeze_path.open("w", encoding="utf-8") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, text=True, check=True)
print("已保存：", freeze_path)
print("环境准备完成。后续 Notebook 请保持相同 PROJECT_ROOT、ARTIFACT_ROOT 与 DATA_ROOT。")